# Short notebook 2 — TDT4173 course project, Tokke–Vinje unit commitment

| | |
|---|---|
| **Kaggle team name** | `[25] Nah, i'd win` |
| **Members** | `[Full name — student ID]` |
| **Members** | `[Full name — student ID]` |
| **Members** | `[Full name — student ID]` |

This notebook reproduces a **logit-space blend of three models**: CatBoost, ExtraTrees and a dilated-convolution sequence network (TCN). The point of building a second notebook from scratch is **diversity**: the goal is to produce predictions that are maximally different from Notebook 1 in model bias, architecture and feature perspective, so that the two `submission.csv` files can be ensembled for a gain. A component that simply re-implements a Notebook 1 model would be wasted effort: averaging two nearly identical models barely helps, while averaging two genuinely different models does.

The exploratory analysis, the feature analysis, all the models we tried, the model interpretation and **how every hyper-parameter value written below was obtained** are in the PDF report. No tuning is done in this notebook.

**The three models, and why each one is here.**

| Model | Family | What it is | Why it is here |
|---|---|---|---|
| **CatBoost** | Gradient boosting (symmetric trees, ordered target statistics) | Builds balanced trees with oblivious decision splits and ordered boosting to reduce prediction shift | Different tree shape, regularisation and target-encoding strategy from XGBoost / LightGBM (Notebook 1) |
| **ExtraTrees** | Pure bagging (randomised trees) | Grows deep trees using **random** split thresholds rather than optimal ones, averaged over the ensemble | No boosting at all; its errors are almost uncorrelated with gradient-boosting errors |
| **TCN (Temporal Convolutional Network)** | Sequence neural network | Reads each unit's 168-hour week through 7 exponentially dilated 1-D convolution blocks, plus a pooled context of the other units | Captures long-range temporal dependencies with a fixed receptive field and residual convolutions, instead of Notebook 1's recurrent GRU + multi-head attention |

### How this notebook differs from Short notebook 1

1. **No shared model families.** Notebook 1 uses XGBoost, LightGBM, a GRU-attention network and a refiner. Notebook 2 uses CatBoost, ExtraTrees and a dilated-convolution network. The only overlap is the raw CSV files.
2. **Feature additions.** Notebook 2 adds calendar features (real weekday / weekend / hour-of-day extracted from timestamps, season via day-of-year), scale-free margin variants (rank and z-score inside the week), and reservoir-group history (filling trend over the last 7 days, mean inflow of the last 7 and 30 days). These give the models a different temporal and hydrological perspective on the same question.
3. **Auxiliary structure loss.** The sequence network is not only trained to predict each unit independently; for the plants with multiple units (Tokke: 4 units, Vinje: 3 units) it is also penalised when the *plant-level average* of the predictions deviates from the plant-level average of the labels. This encourages physically coherent dispatch (if on average 2 of the 4 Tokke units are running, the network should reflect that).
4. **Logit-space blending.** The three models are averaged **in logit space** (i.e. on $\log(p/(1-p))$) rather than in probability space. Tree models and neural-network sigmoids produce probabilities on different calibration scales; blending logits linearly and sigmoiding once is a principled way to combine models with heterogeneous calibration.
5. **Missing-water-value fallback.** At the very end of the dataset (late 2024) the $d+14$ water value does not exist. Notebook 2 replaces it by the $d+7$ value instead of leaving it empty, so the feature pipeline never produces NaNs for structural reasons.

### Changes in this version (same architecture, same three models)
- **Blend fitted inside the notebook.** The previous version used an equal-weight placeholder. Now every model is also trained on the older cases and scored on a temporal hold-out (the most recent 15 % of the training cases, with a 7-day purge so that overlapping weekly runs do not leak). The logit-blend weights are fitted on it (non-negative logistic regression) and compared with equal weights by hold-out AUC. Set `CFG["fit_blend"] = False` to skip this (about +60 % running time) and use the constants in the blend cell instead.
- **Temporal smoothing of the blended logits** (kernel chosen on the same hold-out, `none` if nothing helps).
- **10 extra features for the tree models**: price 1 / 3 / 12 / 24 h before and after, and the margin averaged over the 14 units plus the unit's margin relative to it (the context the network already gets through its context vector).

### How to run
1. Install the packages listed in `requirements.txt` (Python 3.12). CatBoost is required; nothing is skipped silently — a missing package stops the notebook.
2. Place the competition data in the same folder as this notebook, with this layout:
   ```
   ├── short_document_gpt_improved.ipynb
   ├── prediction_mapping.csv
   ├── sample_submission.csv
   └── data/
       ├── kernel/
       └── extended/
   ```
   No internet access and no other file is needed.
3. Run all cells. The prediction is written to `submission.csv` next to the notebook (725 rows × 2353 columns).

### Hardware and running time
*Fill in after a timed run with the numbers printed by the last cell:*
- GPU run: `GPU model, CUDA / PyTorch version`, total `<h>`.
- CPU run (all threads): total `<h>`, peak RAM `<GB>`.

Without a GPU the notebook switches to CPU automatically: CatBoost falls back to CPU, and the TCN is trained with 2 seeds instead of 3 to respect the 12-hour Kaggle limit.

### Reproducibility
All random seeds are fixed (numpy, torch, CatBoost, ExtraTrees). Small differences can come from GPU non-determinism — CatBoost on GPU and GPU convolutions are not bit-exact across CUDA versions — and, on CPU, from the smaller number of network seeds. CPU mode is deterministic up to floating-point order.

## 1. Setup

Imports the libraries and locates the competition files by name in this folder (or its parent), so no path has to be edited. It also selects the GPU when available, otherwise the CPU, uses all CPU threads, and fixes every random seed (numpy, torch, CatBoost, ExtraTrees).

Every number that controls training lives in the configuration dict `CFG`: the number of boosting rounds, tree depth and learning rate for CatBoost, the row sub-sample and tree count for ExtraTrees, the width, epochs and number of seeds of the sequence network, the weight of the auxiliary loss, and the size and purge of the temporal hold-out used to fit the blend. These values were chosen with temporal cross-validation during development, **not** in this notebook.

The output file is `submission.csv`.

In [ ]:
import os, time, random, warnings, contextlib
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import torch
import torch.nn.functional as tF
from catboost import CatBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize
warnings.filterwarnings("ignore")

def find_file(name, max_depth=4):
    # locate an original competition file below this folder (or its parent), by file name
    for root in (Path.cwd(), Path.cwd().parent):
        for depth in range(max_depth + 1):
            hits = sorted(p for p in root.glob("/".join(["*"] * depth + [name]))
                          if not any(part.startswith(".") for part in p.relative_to(root).parts))
            if hits:
                return hits[0]
    raise FileNotFoundError(f"{name} not found: place the competition data next to this notebook")

KERNEL      = find_file("Unit_commitment_decisions.csv").parent
EXTENDED    = find_file("Tokke_Vinje_topology.yaml").parent
MAPPING_CSV = find_file("prediction_mapping.csv")
SAMPLE_CSV  = find_file("sample_submission.csv")
OUT_CSV     = Path("submission.csv")

SEED = 0
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
N_THREADS = os.cpu_count() or 4
torch.set_num_threads(N_THREADS)
seed_everything(SEED)

# Every hyper-parameter lives here. They were chosen with temporal cross-validation (dev notebook).
CFG = dict(
    cb_iters=800, cb_depth=8, cb_lr=0.05,                 # CatBoost
    et_rows=1_500_000, et_trees=200, et_leaf=30,          # ExtraTrees (row sub-sample to bound memory)
    tcn_epochs=12, tcn_dim=64, tcn_seeds=3 if DEVICE == "cuda" else 2,
    aux_w=0.5,                                            # weight of the plant-average auxiliary loss
    fit_blend=True, hold_frac=0.15, purge_days=7,         # temporal hold-out that fits the blend weights and the smoothing
)
T0 = time.time()
print("device:", DEVICE, "| threads:", N_THREADS)

## 2. Data and hydro topology

Reads the original competition files: reservoir volumes, water values, inflows, day-ahead prices, the environmental constraints (minimum flow and minimum volume), the topology of the system, the training labels (`Unit_commitment_decisions.csv`) and the list of test cases (`prediction_mapping.csv`). It then builds two structures:

- `cases`: a table with one row per case (a one-week SHOP run starting on a given day), with its `Run No`, start date and a train/test flag. It lists the 2,922 training cases (2015–2022) followed by the 725 test cases (2023–2024), in the exact submission order.
- `labels`: a 3D array of shape (2,922 cases, 14 units, 168 hours) with the commitment status (1 = on, 0 = off) of the training cases. In the CSV each case is a single row of 14 × 168 columns; here it is reshaped into one 14 × 168 grid per case.

It also builds the topology mappings that tell the feature engineering step which reservoirs to look at for each unit: the reservoir feeding each plant (`UP`), the reservoir receiving its discharge (`DOWN`), the reservoirs joined by tunnels at the same level that act as a single storage (`UP_GROUP`), and each unit's maximum discharge $Q_{\max}$ and maximum power $P_{\max}$, taken from the topology file.

In the feature engineering step, `cases` gives the start date of each week, which is used to collect that week's prices, inflows, volumes and water values and to compute one row of features for every (case, unit, hour). `labels`, flattened in the same order, gives the 0/1 target of each training row, which is what the models learn to predict.

In [ ]:
H = 168
read = lambda p: pd.read_csv(p, parse_dates=["date"]).set_index("date")

volume   = read(KERNEL / "Historical_volume_2015_2024.csv")
wv       = read(KERNEL / "Synthetic_water_value_2015_2024.csv")
inflow   = read(KERNEL / "Historical_inflow_1958_2025.csv")
price_s  = read(KERNEL / "Historical_day_ahead_price_2015_2025.csv").iloc[:, 0]
min_flow = read(EXTENDED / "Constraint_min_flow.csv")
min_vol  = read(EXTENDED / "Constraint_min_volume.csv")
uc       = pd.read_csv(KERNEL / "Unit_commitment_decisions.csv")
mapping  = pd.read_csv(MAPPING_CSV)
topo     = yaml.safe_load(open(EXTENDED / "Tokke_Vinje_topology.yaml"))["model"]

UNITS = ["Hogga_G1", "Tokke_G1", "Tokke_G2", "Tokke_G3", "Tokke_G4", "Lio_G1", "Byrte_G1",
         "Vinje_G1", "Vinje_G2", "Vinje_G3", "Haukeli_G1", "Songa_G1", "Kjela_G1", "Vesle Kjela_G1"]
U = len(UNITS)
PLANT = [u.rsplit("_", 1)[0] for u in UNITS]
LABEL_COLS = [f"result_committed_{u}_t{t}" for u in UNITS for t in range(H)]

# One row per case (a one-week run starting on a given day): train cases first, then test cases in submission order
cases = pd.concat([uc[["Run No", "starttime"]].assign(is_train=True),
                   mapping[["Run No", "starttime"]].assign(is_train=False)], ignore_index=True)
cases["start"] = pd.to_datetime(cases["starttime"]).dt.tz_localize("UTC")
n_cases, n_train = len(cases), int(cases.is_train.sum())
labels = uc[LABEL_COLS].to_numpy(np.int8).reshape(n_train, U, H)      # (case, unit, hour), 1 = committed
print(f"{n_train} train cases, {n_cases - n_train} test cases, ON rate {labels.mean():.3f}")

# Hydro topology (from the connections in the topology file)
UP = {"Hogga": ["Bandak"], "Tokke": ["Vinjevatn"], "Lio": ["Byrtevatn"], "Byrte": ["Botnedalsvatn"],
      "Vinje": ["Vaamarvatn"], "Haukeli": ["Vatjern"], "Songa": ["Songavatn", "Bitdalsvatn"],
      "Kjela": ["Foersvatn"], "Vesle Kjela": ["Kjelavatn"]}
DOWN = {"Hogga": [], "Tokke": ["Bandak"], "Lio": ["Bandak"], "Byrte": ["Byrtevatn"],
        "Vinje": ["Vinjevatn"], "Haukeli": ["Vinjevatn"], "Songa": ["Totak"],
        "Kjela": ["Hyljelihyl"], "Vesle Kjela": ["Foersvatn"]}
UP_GROUP = {"Hogga": ["Bandak"], "Tokke": ["Vinjevatn"], "Lio": ["Byrtevatn"], "Byrte": ["Botnedalsvatn"],
            "Vinje": ["Vaamarvatn", "Totak", "Venemo"], "Haukeli": ["Vatjern", "Langeidvatn"],
            "Songa": ["Songavatn", "Bitdalsvatn"], "Kjela": ["Foersvatn", "Bordalsvatn"],
            "Vesle Kjela": ["Kjelavatn", "Staavatn"]}
MAX_VOL = {k: v["max_vol"] for k, v in topo["reservoir"].items()}
GEN = topo["generator"]
QMAX = {u: max(max(c["x"]) for c in GEN[u]["turb_eff_curves"]) for u in UNITS}
LIO_FLOW = ["b_Byrtevatn", "r_Vest_Vassdraget"]
MIN_VOL_MARGIN = ["Totak", "Byrtevatn"]

## 3. Feature table

The models always answer the same question: *"in this week, is this unit ON at this hour?"*. Each row of the feature table therefore corresponds to one of these questions, to one combination of:

- **case**: which week (e.g. the one starting on 1 January 2015);
- **unit**: which generator (e.g. Tokke_G1);
- **hour**: which of the 168 hours of the week (e.g. hour 40).

| case | unit | hour | price | margin m | g_fill | … (features) | target |
|---|---|---:|---:|---:|---:|---|---:|
| 2015-01-01 | Hogga_G1 | 0 | 30.1 | 2.4 | 0.61 | … | 1 |
| 2015-01-01 | Hogga_G1 | 1 | 29.8 | 2.1 | 0.61 | … | 1 |
| … | … | … | … | … | … | … | … |
| 2015-01-01 | Hogga_G1 | 167 | … | … | … | … | 0 |
| 2015-01-01 | Tokke_G1 | 0 | 30.1 | 5.7 | 0.83 | … | 1 |
| … | … | … | … | … | … | … | … |
| 2015-01-02 | Hogga_G1 | 0 | … | … | … | … | … |

*(values are illustrative)*

- Each case produces 14 units × 168 hours = 2,352 rows; in total 3,647 cases × 2,352 ≈ **8.6 million rows**.
- The table contains both the 2,922 training cases (≈ 6.9 million rows) and, after them, the 725 test cases (≈ 1.7 million rows), with features computed in exactly the same way. Only the training rows have a 0/1 target, kept in a separate array (`labels` flattened in the same order); the test rows are the ones the models must predict.
- The tree models (CatBoost, ExtraTrees) look at one row at a time and learn to map its features to the target. The sequence network instead receives a whole case at once, as a grid of 14 units × 168 hours × features, and predicts the on/off probability of every cell of the grid together (Section 4).

### Main feature groups

The groups below follow the comments inside `build_features`. The full analysis of which groups are useful is in the PDF report. Features that exist only in Notebook 2 (not in Notebook 1) are marked.

| Group | What it describes | Examples |
|---|---|---|
| **Calendar / season** | Real timestamps: hour-of-day, weekday vs weekend, day-of-year encoded as sine/cosine. Notebook 1 only used the position in the week (`t`) and a unit index. | `hour_of_day`, `weekday`, `is_weekend`, `doy_sin`, `doy_cos` |
| **Price shape** | Hourly price, intra-week rank, intra-day rank, rolling means, volatility, acceleration. | `price`, `p_rank`, `p_z`, `p_c6`, `p_c24`, `p_vol6`, `p_accel`, `p_day_rank` |
| **Per-unit economics** | Margin $m = \text{price} - \text{WV}_{d+7}(\text{upstream})$. This is the single most informative quantity: the unit tends to be on when $m > 0$. | `m`, `m_cmean7`, `m_cmean25`, `m_frac_week`, `m_dprev`, `m_dnext`, `m_wq20`, `m_wq80` |
| **Scale-free margin** | Rank and z-score of the margin inside the week; interaction of margin with price rank. Different price levels across years are normalised away. | `m_rank`, `m_z`, `m_x_prank`, `m_over_pstd` |
| **Reservoir group state** | Fill level, spill risk, headroom in days, hourly inflow, 24-hour centred mean inflow, aggregated over the tunnel-connected reservoir group (`UP_GROUP`). | `g_fill`, `g_spill_risk`, `g_headroom_days`, `g_inflow_h`, `g_inflow_c24` |
| **Recent reservoir history** | Change in filling over the last week; mean inflow in the 7 and 30 days *before* the case starts (the inflow file starts in 1958, so it always exists). | `u_dV7`, `u_inflow_prev7`, `u_inflow_prev30` |
| **System-level water** | Total inflow and total stored volume across *all* reservoirs, giving a catchment-wide wet/dry context. | `sys_total_inflow`, `sys_total_fill` |
| **Hydro budget** | Water available, expressed as hours of full production (storage ÷ max discharge). | `hb_storage_local_h`, `hb_inflow_local_h`, `hb_headroom_local_h`, `hb_spill_local_h` |
| **Unit data** | Max power, start-up cost normalised by $P_{\max}$, plant size, unit index within plant. | `gen_pmax`, `gen_start_per_mw`, `plant_n_units` |
| **Environmental constraints** | Hourly minimum flow on Byrtevatn and Vest Vassdraget; deficit that Lio must cover; minimum volume margins. | `mfh_*`, `rvest_deficit_h`, `mvmargin_*` |

**Missing-value handling for $d+14$ water values.** At the end of 2024 the $d+14$ water value does not exist. It is replaced by the $d+7$ value so the feature pipeline never produces NaNs for structural reasons.

**Added in this version (tree models; the network already sees these through its convolutions and context vector):** `p_lag{1,3,12,24}` / `p_lead{1,3,12,24}` (price 1, 3, 12 and 24 hours before / after — a tree sees one row at a time and cannot look at neighbouring hours by itself), `m_units_mean` (margin averaged over the 14 units at this hour) and `m_minus_units` (this unit's margin minus that average — the same cross-unit context the network receives through its context vector).

The output is a single float32 matrix `X` with one row per (case, unit, hour), rows ordered CASE → UNIT → HOUR (= submission order), and the list `FEATURES` of column names. Training rows come first: `X[:n_tr_rows]` is the training part, `X[n_tr_rows:]` the test part.

Runs the feature construction on all cases (training and test). `X` has one row per (case, unit, hour); the training rows come first, so `X[:n_tr_rows]` is a view of the training part and `X[n_tr_rows:]` the test part. `y_tr` is the 0/1 target of the training rows, in the same order. After construction, the cell prints the shape, the memory footprint, and any feature that still has missing values (expected: none, because of the $d+14$ fallback).

In [ ]:
# Helper functions for aggregation and rolling statistics along the 168-hour week --------------------------
def _per_unit(mapping, values, agg):
    """values: reservoir -> array (n,). Returns (n, U) aggregated over the reservoirs linked to each unit's plant."""
    out = []
    for p in PLANT:
        rs = mapping[p]
        out.append(agg(np.stack([values[r] for r in rs]), 0) if rs
                   else np.zeros_like(next(iter(values.values()))))
    return np.stack(out, 1)

def _rolling(a, w, fn):          # centred rolling statistic along the 168 hours
    n, u, h = a.shape
    s = pd.DataFrame(a.reshape(-1, h).T).rolling(w, center=True, min_periods=1)
    return getattr(s, fn)().to_numpy().T.reshape(n, u, h)

def _run_lengths(above):         # hours since the last / until the next sign change of a true-false series
    dp = np.zeros(above.shape, np.float32); dn = np.zeros(above.shape, np.float32)
    for t in range(1, H):
        dp[..., t] = np.where(above[..., t] == above[..., t - 1], dp[..., t - 1] + 1, 0)
    for t in range(H - 2, -1, -1):
        dn[..., t] = np.where(above[..., t] == above[..., t + 1], dn[..., t + 1] + 1, 0)
    return dp, dn

def _margin_context(m, prefix, f):
    """Adds a margin m and its rich temporal context to dict f: rolling stats, share of positive hours,
    distance to sign changes, weekly quantiles, positive/negative sums."""
    f[prefix] = m
    for w in (7, 25):
        f[f"{prefix}_cmean{w}"] = _rolling(m, w, "mean")
    above = m > 0
    f[f"{prefix}_frac_week"] = above.mean(-1, keepdims=True)
    f[f"{prefix}_dprev"], f[f"{prefix}_dnext"] = _run_lengths(above)
    f[f"{prefix}_wq20"] = np.percentile(m, 20, axis=-1, keepdims=True)
    f[f"{prefix}_wq80"] = np.percentile(m, 80, axis=-1, keepdims=True)
    f[f"{prefix}_pos_sum"] = np.clip(m, 0, None).sum(-1, keepdims=True)

# ----------------------------------------------------------------------------------------------------------
def build_features(cases):
    st = pd.DatetimeIndex(cases["start"]); n = len(st)
    # Every hour of every case as a real UTC timestamp (needed for calendar features)
    hours = pd.DatetimeIndex((st.values[:, None] + np.arange(H) * np.timedelta64(1, "h")).ravel(), tz="UTC")
    hourly = lambda s: s.reindex(hours).to_numpy().reshape(n, H)
    res = list(volume.columns)
    price = hourly(price_s); assert not np.isnan(price).any()
    f = {}
    C = lambda a: np.asarray(a, np.float32).reshape(n, 1, 1)     # case-level broadcast
    T = lambda a: np.asarray(a, np.float32).reshape(n, 1, H)     # case × hour broadcast
    day = lambda d: pd.Timedelta(days=d)

    # ---- Reservoir-level raw values ---------------------------------------------------------------
    V0  = {r: volume[r].reindex(st).to_numpy() for r in res}
    Vm7 = {r: volume[r].reindex(st - day(7)).to_numpy() for r in res}
    W0  = {r: wv[r].reindex(st).to_numpy() for r in res}
    W7  = {r: wv[r].reindex(st + day(7)).to_numpy() for r in res}
    W14 = {r: wv[r].reindex(st + day(14)).to_numpy() for r in res}
    # Fallback: after the end of 2024 W14 is missing → reuse W7
    W14 = {r: np.where(np.isnan(W14[r]), W7[r], W14[r]) for r in res}
    inflow_h = {r: hourly(inflow[r]) for r in res}
    # Inflow of the 7 / 30 days BEFORE the case starts (the inflow file starts in 1958, so it always exists)
    prev = st - pd.Timedelta(hours=1)
    I7p  = {r: inflow[r].rolling(24 * 7,  min_periods=24).mean().reindex(prev).to_numpy() for r in res}
    I30p = {r: inflow[r].rolling(24 * 30, min_periods=24 * 7).mean().reindex(prev).to_numpy() for r in res}

    # ---- Price shape of the week ------------------------------------------------------------------
    f["p_std"] = C(price.std(1)); f["p_q90"] = C(np.percentile(price, 90, axis=1))
    f["p_q10"] = C(np.percentile(price, 10, axis=1)); f["p_spread"] = f["p_q90"] - f["p_q10"]
    f["p_peak_hour"] = C(np.argmax(price, 1).astype(np.float32))
    f["price"] = T(price); f["p_rank"] = T(price.argsort(1).argsort(1) / (H - 1))
    f["p_minus_mean"] = T(price - price.mean(1, keepdims=True))
    f["p_z"] = T((price - price.mean(1, keepdims=True)) / (price.std(1, keepdims=True) + 1e-3))
    for w in (6, 24):
        f[f"p_c{w}"] = T(pd.DataFrame(price.T).rolling(w, center=True, min_periods=1).mean().to_numpy().T)
    f["p_accel"] = T(np.gradient(np.gradient(price, axis=1), axis=1))
    f["p_vol6"] = T(pd.DataFrame(price.T).rolling(6, min_periods=1).std().fillna(0).to_numpy().T)
    # Intra-day rank: reorder each day independently, then flatten back to 168 hours
    f["p_day_rank"] = T(price.reshape(n, 7, 24).argsort(-1).argsort(-1).reshape(n, H) / 23)
    # Neighbouring prices (a tree only sees one row; the network sees the neighbours through its convolutions)
    pp = np.pad(price, ((0, 0), (24, 24)), mode="edge")
    for k in (1, 3, 12, 24):
        f[f"p_lag{k}"] = T(pp[:, 24 - k:24 - k + H]); f[f"p_lead{k}"] = T(pp[:, 24 + k:24 + k + H])

    # ---- Calendar features (real timestamps, NOT just t=0..167) ----------------------------------
    f["hour_of_day"] = T(hours.hour.to_numpy().reshape(n, H))
    dow = hours.dayofweek.to_numpy().reshape(n, H)
    f["weekday"] = T(dow)
    f["is_weekend"] = T(dow >= 5)                          # Saturday=5, Sunday=6
    doy = st.dayofyear.to_numpy()
    f["doy_sin"] = C(np.sin(2 * np.pi * doy / 365.25))   # Seasonal cycle
    f["doy_cos"] = C(np.cos(2 * np.pi * doy / 365.25))
    f["t"] = np.arange(H, dtype=np.float32).reshape(1, 1, H)
    f["unit"] = np.arange(U, dtype=np.float32).reshape(1, U, 1)

    # ---- Per-unit economics from the topology -----------------------------------------------------
    for name, d, agg in (("W7", W7, np.mean), ("W0", W0, np.mean), ("W14", W14, np.mean), ("V0", V0, np.sum),
                         ("Iwk", {r: inflow_h[r].mean(1) for r in res}, np.sum)):
        f[f"u_up_{name}"] = _per_unit(UP, d, agg)[:, :, None]
        f[f"u_dn_{name}"] = _per_unit(DOWN, d, agg)[:, :, None]
    f["u_netW7"] = f["u_up_W7"] - f["u_dn_W7"]
    f["u_netW14"] = f["u_up_W14"] - f["u_dn_W14"]
    P = price[:, None, :]
    m = P - f["u_up_W7"]                                  # core margin: price - water value(d+7) upstream
    _margin_context(m, "m", f)
    f["m14"] = P - f["u_up_W14"]
    # Scale-free margin variants: robust to absolute price level across years
    f["m_rank"] = m.argsort(-1).argsort(-1).astype(np.float32) / (H - 1)
    f["m_z"] = (m - m.mean(-1, keepdims=True)) / (m.std(-1, keepdims=True) + 1e-3)
    f["m_x_prank"] = m * f["p_rank"]
    f["m_over_pstd"] = m / (price.std(1)[:, None, None] + 1)
    f["u_ratio"] = P / (np.abs(f["u_up_W7"]) + 1)
    f["m_units_mean"] = np.broadcast_to(m.mean(1, keepdims=True), m.shape)     # what the 14 units see on average at this hour
    f["m_minus_units"] = m - f["m_units_mean"]

    # ---- Reservoir state, spill risk, inflow, recent history -------------------------------------
    Vg = _per_unit(UP_GROUP, V0, np.sum); Vg7 = _per_unit(UP_GROUP, Vm7, np.sum)
    capg = _per_unit(UP_GROUP, {r: np.full(n, MAX_VOL[r]) for r in res}, np.sum)
    Ig = _per_unit(UP_GROUP, {r: inflow_h[r].sum(1) * 3600 / 1e6 for r in res}, np.sum)
    f["g_fill"] = (Vg / capg)[:, :, None]
    f["g_inflow_over_cap"] = (Ig / capg)[:, :, None]
    f["g_spill_risk"] = ((Vg + Ig - capg) / capg)[:, :, None]
    f["g_headroom_days"] = ((capg - Vg) / (Ig / 7 + 1e-3))[:, :, None]
    Ih = _per_unit(UP_GROUP, inflow_h, np.sum)
    f["g_inflow_h"] = Ih
    f["g_inflow_c24"] = _rolling(Ih, 25, "mean")
    # Filling trend over the last week
    f["u_dV7"] = ((Vg - Vg7) / capg)[:, :, None]
    # Convert mean inflow (m3/s) over d days to Mm3, then normalise by capacity
    to_mcm = lambda q, d: {r: q[r] * 86400 * d / 1e6 for r in res}
    f["u_inflow_prev7"]  = (_per_unit(UP_GROUP, to_mcm(I7p, 7),   np.sum) / capg)[:, :, None]
    f["u_inflow_prev30"] = (_per_unit(UP_GROUP, to_mcm(I30p, 30), np.sum) / capg)[:, :, None]
    f["sys_total_inflow"] = C(np.stack([inflow_h[r].sum(1) for r in res], 0).sum(0))
    f["sys_total_fill"] = C(np.stack([V0[r] for r in res], 0).sum(0))

    # ---- Unit static data ------------------------------------------------------------------------
    f["gen_pmax"] = np.array([GEN[u]["p_max"] for u in UNITS], np.float32).reshape(1, U, 1)
    f["gen_start_per_mw"] = np.array([GEN[u]["startcost_const"] / GEN[u]["p_max"] for u in UNITS],
                                     np.float32).reshape(1, U, 1)
    f["plant_n_units"] = np.array([PLANT.count(p) for p in PLANT], np.float32).reshape(1, U, 1)
    f["unit_idx_in_plant"] = np.array([int(u.rsplit("_G", 1)[1]) for u in UNITS], np.float32).reshape(1, U, 1)
    f["m_minus_startcost"] = f["m_cmean7"] - f["gen_start_per_mw"] / 7

    # ---- Hydro budget: water expressed as hours of full production -------------------------------
    q_plant = np.array([sum(QMAX[u] for u in UNITS if u.startswith(p + "_")) for p in PLANT])
    q_unit = np.array([QMAX[u] for u in UNITS])
    to_h = lambda mcm, q: (mcm * 1e6 / (q[None, :] * 3600))[:, :, None]
    f["hb_storage_local_h"] = to_h(Vg, q_plant)
    f["hb_inflow_local_h"] = to_h(Ig, q_plant)
    f["hb_headroom_local_h"] = to_h(capg - Vg, q_plant)
    f["hb_spill_local_h"] = f["hb_inflow_local_h"] - f["hb_headroom_local_h"]
    f["hb_inflow_unit_h"] = to_h(Ig, q_unit)

    # ---- Environmental constraints ---------------------------------------------------------------
    mf = {c: hourly(min_flow[c]) for c in LIO_FLOW}
    for c, v in mf.items():
        f[f"mfh_{c}"] = T(v)
    deficit = mf["r_Vest_Vassdraget"] - hourly(inflow["r_Vest_Vassdraget"])
    f["rvest_deficit_h"] = T(deficit)
    f["rvest_deficit_over_lio"] = T(np.clip(deficit, 0, None) / QMAX["Lio_G1"])
    for c in MIN_VOL_MARGIN:
        f[f"mvmargin_{c}"] = C(V0[c] - hourly(min_vol[c]).max(1))
    f["mvh_Byrtevatn"] = T(hourly(min_vol["Byrtevatn"]))

    # ---- Flatten into one float32 matrix, rows ordered CASE → UNIT → HOUR ------------------------
    names = list(f)
    X = np.empty((n * U * H, len(names)), np.float32)
    for j, k in enumerate(names):
        X[:, j] = np.broadcast_to(np.asarray(f.pop(k), np.float32), (n, U, H)).ravel()
    return X, names

t = time.time()
X, FEATURES = build_features(cases)
n_tr_rows = n_train * U * H                  # train rows come first
y_tr = labels.ravel()

# Optional: drop features if a dev-notebook ablation says they hurt
DROP_FEATURES = []
assert set(DROP_FEATURES) <= set(FEATURES), "unknown feature name in DROP_FEATURES"
if DROP_FEATURES:
    keep = [i for i, k in enumerate(FEATURES) if k not in DROP_FEATURES]
    X = X[:, keep].copy(); FEATURES = [FEATURES[i] for i in keep]

print(f"features: {X.shape}, {X.nbytes / 1e9:.1f} GB, {time.time() - t:.0f}s")
nan_frac = np.isnan(X).mean(0)
print("features with missing values:", {k: round(float(v), 4) for k, v in zip(FEATURES, nan_frac) if v > 0} or "none")

## 4. Models

The three model families are defined here. They all consume the same feature table `X`, but the sequence network receives a standardised copy arranged as a $(\text{case}, \text{unit}, \text{hour}, \text{features})$ grid (Section 5). Each family was chosen to be as different as possible from the corresponding Notebook 1 model, because blend diversity is the whole point of this notebook.

### CatBoost — symmetric trees with ordered boosting

**Idea.** XGBoost (Notebook 1) builds trees level-wise up to a maximum depth; LightGBM grows leaf-wise, always splitting the leaf that reduces the error the most. CatBoost builds **symmetric (oblivious) trees**, where the same split condition is applied to every node at a given depth. This gives a very different inductive bias: the model is more regularised and less prone to overfitting on spurious deep interactions. In addition, its **ordered target statistics** compute each categorical encoding using only preceding examples (ordered boosting), which reduces the prediction shift that affects standard greedy target encoding.

**Training and hyper-parameters.** 800 rounds, depth 8, learning rate 0.05, L2 leaf regularisation 5, Bernoulli row sub-sampling 0.8 and 128 quantization bins. On GPU `task_type="GPU"` is used (column sub-sampling `rsm` is unavailable on GPU, so it is only set on CPU, together with all CPU threads). These values were chosen with temporal cross-validation in the dev notebook; no tuning is done here.

**Why it is used.** Its errors are structurally different from XGBoost's and LightGBM's — different tree shape, different regularisation, different encoding strategy — which is exactly what a blend wants.

### ExtraTrees — pure bagging with random split thresholds

**Idea.** ExtraTrees (Extremely Randomized Trees) does **not boost**; it averages many deep trees. At each split it picks a threshold *uniformly at random* between the minimum and maximum of the feature, not the optimal threshold, and each tree only sees a random half of the features (`max_features=0.5`). Each single tree is therefore weak and unstable, but the trees are much more independent than boosted trees, and the ensemble variance drops purely by averaging. The rows are sub-sampled (1.5 million random training rows) purely to bound memory.

**Training and hyper-parameters.** 200 trees, `min_samples_leaf=30`, trained on the row sub-sample; missing values are replaced by the training median, and predictions are produced in chunks of 500,000 rows to limit memory.

**Why it is used.** Because it is not a gradient-boosting model, the mistakes it makes are almost orthogonal to CatBoost's mistakes, which is precisely what we want in a blend.

### TCN (Temporal Convolutional Network) — dilated convolutions instead of recurrence

**Idea.** Notebook 1's network combines a bidirectional GRU (recurrence over the week) with multi-head attention (across units). Our network replaces both with **dilated causal convolutions**:

- Each of the 7 blocks has kernel size 3 and dilation $2^{\text{block}}$ ($1, 2, 4, \dots, 64$). The receptive field therefore covers $1 + 2 \times (1+2+4+\dots+64) = 255$ hours, i.e. the whole 168-hour week and more.
- Unlike an RNN, a TCN processes all time steps in parallel in each layer, and the residual dilated structure avoids vanishing gradients.
- Cross-unit context is provided not by attention but by a **global context vector**: after the convolutions, the hidden states are pooled across all units at the same hour, projected, and added back to every unit. This tells each unit "what the other units are doing now" without the quadratic cost of attention.

**Architecture.** Each cell (unit, hour) is projected to a vector of size `d=64` and a learned unit code is added; then 7 residual blocks are applied, each consisting of a dilated Conv1d → GELU → dropout → pointwise Conv1d, added back to the input and normalised with GroupNorm. The pooled cross-unit context is concatenated with each cell's representation and passed through a small feed-forward head (Linear → GELU → Linear) that produces one logit per (unit, hour).

**Auxiliary loss for multi-unit plants.** Tokke (4 units) and Vinje (3 units) share the same upstream water value and price. The commitment pattern is therefore constrained: the *average* number of running units follows the plant-level economics more cleanly than the commitment of any single unit. The network is therefore penalised, in addition to the per-unit binary cross-entropy, when the plant-level average of its predictions deviates from the plant-level average of the labels:

$$\mathcal{L}_{\text{aux}} = \text{BCE}\left(\frac{1}{|\text{plant}|}\sum_{u \in \text{plant}} \sigma(z_u), \; \frac{1}{|\text{plant}|}\sum_{u \in \text{plant}} y_u\right)$$

with weight `CFG["aux_w"] = 0.5`. This encourages physically consistent dispatch across units of the same plant.

**Training and hyper-parameters.** Binary cross-entropy loss, AdamW optimiser (learning rate 2e-3 with a one-cycle schedule, weight decay 1e-3), dropout 0.2, batches of 16 cases, 12 epochs, gradient clipping at 1.0, and bf16 mixed precision on GPU. The predictions of several seeds are averaged (3 on GPU, 2 on CPU) to reduce variance.

**Why it is used.** On its own the network is weaker than the tree models, but its errors are quite different from theirs, so adding it to the blend gives the largest gain among all components — and its convolutive temporal bias is different from Notebook 1's recurrent network, which matters for the final inter-notebook ensemble.

In [ ]:
def fit_predict_catboost(X_tr, y_tr, X_te, iters, seed=SEED, eval_set=None):
    """Train a CatBoost classifier on the feature table and return test probabilities."""
    params = dict(iterations=iters, depth=CFG["cb_depth"], learning_rate=CFG["cb_lr"], l2_leaf_reg=5,
                  bootstrap_type="Bernoulli", subsample=0.8, border_count=128,
                  random_seed=seed, verbose=0, allow_writing_files=False, loss_function="Logloss")
    if DEVICE == "cuda":
        params["task_type"] = "GPU"                      # GPU mode; column sub-sampling (rsm) unavailable on GPU
    else:
        params.update(thread_count=N_THREADS, rsm=0.8)
    if eval_set is not None:
        params["eval_metric"] = "AUC"
    clf = CatBoostClassifier(**params)
    clf.fit(X_tr, y_tr, eval_set=eval_set, use_best_model=False)
    return clf.predict_proba(X_te)[:, 1].astype(np.float32), clf

def fit_predict_et(X_tr, y_tr, X_te, seed=SEED):
    """Train ExtraTrees on a random row sub-sample (memory bound) and predict in chunks."""
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(len(X_tr), min(CFG["et_rows"], len(X_tr)), replace=False))
    Xs, ys = X_tr[idx], y_tr[idx]
    med = np.nan_to_num(np.nanmedian(Xs, axis=0)).astype(np.float32)      # missing values → train median
    fill = lambda a: np.where(np.isnan(a), med, a)
    clf = ExtraTreesClassifier(n_estimators=CFG["et_trees"],
                               min_samples_leaf=CFG["et_leaf"],
                               max_features=0.5,            # each tree sees half the features
                               n_jobs=N_THREADS,
                               random_state=seed,
                               bootstrap=False)             # no bootstrap: pure sub-sample + random splits
    clf.fit(fill(Xs), ys)
    out = np.empty(len(X_te), np.float32)
    for i in range(0, len(X_te), 500_000):                  # chunking limits memory
        out[i:i + 500_000] = clf.predict_proba(fill(X_te[i:i + 500_000]))[:, 1]
    return out

# ---- Sequence network input prep -------------------------------------------------------------
def prep_tensor(X, n_cases, n_stat_rows):
    """Standardise with training statistics, clip, flag NaNs with indicator columns. float16 to save memory.
    Returns (n_cases, U, H, F')."""
    nf = X.shape[1]
    S = X[:n_stat_rows:7]                                    # stride 7: ~one point per week, fast stats
    mean = np.nanmean(S, 0); std = np.nanstd(S, 0) + 1e-6
    has_nan = np.zeros(nf, bool)
    for i in range(0, len(X), 1_000_000):
        has_nan |= np.isnan(X[i:i + 1_000_000]).any(0)
    nan_cols = np.flatnonzero(has_nan)
    Z = np.empty((len(X), nf + len(nan_cols)), np.float16)
    for i in range(0, len(X), 500_000):
        x = X[i:i + 500_000]; nan = np.isnan(x)
        z = np.clip((x - mean) / std, -6, 6); z[nan] = 0
        Z[i:i + 500_000, :nf] = z; Z[i:i + 500_000, nf:] = nan[:, nan_cols]
    return Z.reshape(n_cases, U, H, -1)

# ---- Plant-level averaging matrix for auxiliary loss -----------------------------------------
PLANTS = list(dict.fromkeys(PLANT))
MULTI = [p for p in PLANTS if PLANT.count(p) > 1]
PM = np.zeros((U, len(MULTI)), np.float32)
for i, p in enumerate(PLANT):
    if p in MULTI:
        PM[i, MULTI.index(p)] = 1.0
PM /= PM.sum(0, keepdims=True)                             # PM[u, plant] = 1 / n_units(plant)

# ---- TCN architecture ------------------------------------------------------------------------
class Block(torch.nn.Module):
    """Residual dilated convolution block: GroupNorm + GELU + Dropout."""
    def __init__(self, d, dil, dropout):
        super().__init__()
        nn = torch.nn
        self.c1 = nn.Conv1d(d, d, 3, padding=dil, dilation=dil)
        self.c2 = nn.Conv1d(d, d, 1)
        self.norm = nn.GroupNorm(1, d)                      # layer-norm-like per channel
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.norm(x + self.c2(self.drop(tF.gelu(self.c1(x)))))

class TCN(torch.nn.Module):
    """Dilated convolution sequence network.
    Input: (B, U, H, F). Output: (B, U, H) logits."""
    def __init__(self, n_feat, d=64, dilations=(1, 2, 4, 8, 16, 32, 64), dropout=0.2):
        super().__init__()
        nn = torch.nn
        self.d = d
        self.inp = nn.Sequential(nn.Linear(n_feat, d), nn.GELU(), nn.Dropout(dropout))
        self.unit_emb = nn.Embedding(U, d)                  # each unit learns a code
        self.blocks = nn.ModuleList(Block(d, k, dropout) for k in dilations)
        self.ctx = nn.Linear(d, d)                          # projects pooled unit context
        self.head = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Linear(d, 1))
    def forward(self, x):                                     # x: (B, U, H, F)
        B = x.shape[0]
        h = self.inp(x) + self.unit_emb.weight[None, :, None, :]
        # Convolve along the 168 hours of each unit independently
        t = h.reshape(B * U, H, self.d).transpose(1, 2)    # (B*U, d, H)
        for blk in self.blocks:
            t = blk(t)
        h = t.transpose(1, 2).reshape(B, U, H, self.d)
        # Context: average over units at the same hour, project, broadcast back
        ctx = self.ctx(h.mean(1, keepdim=True)).expand(-1, U, -1, -1)
        return self.head(torch.cat([h, ctx], -1)).squeeze(-1)  # (B, U, H)

def fit_predict_tcn(Z, y3, tr_idx, te_idx, seed, epochs, y_te=None):
    """Train the TCN and return predictions for te_idx cases."""
    seed_everything(seed)
    rng = np.random.default_rng(seed)
    model = TCN(Z.shape[-1], CFG["tcn_dim"]).to(DEVICE)
    BS = 16
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-3)
    steps_per_epoch = int(np.ceil(len(tr_idx) / BS))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=steps_per_epoch * epochs)
    pm = torch.from_numpy(PM).to(DEVICE)
    amp = (torch.autocast(device_type="cuda", dtype=torch.bfloat16)
           if DEVICE == "cuda" else contextlib.nullcontext())

    def predict():
        model.eval(); out = []
        with torch.no_grad():
            for i in range(0, len(te_idx), 32):
                xb = torch.from_numpy(Z[te_idx[i:i + 32]]).to(DEVICE).float()
                with amp:
                    lg = model(xb)
                out.append(torch.sigmoid(lg.float()).cpu().numpy())
        return np.concatenate(out).astype(np.float32)

    for ep in range(epochs):
        model.train()
        perm = rng.permutation(tr_idx)
        for i in range(0, len(perm), BS):
            b = np.sort(perm[i:i + BS])
            xb = torch.from_numpy(Z[b]).to(DEVICE).float()
            yb = torch.from_numpy(y3[b]).to(DEVICE).float()
            with amp:
                logits = model(xb)
            logits = logits.float()
            loss = tF.binary_cross_entropy_with_logits(logits, yb)
            # Auxiliary loss: plant-level average must match
            if CFG["aux_w"] > 0 and len(MULTI):
                pa = torch.einsum("buh,up->bph", torch.sigmoid(logits), pm).clamp(1e-6, 1 - 1e-6)
                ya = torch.einsum("buh,up->bph", yb, pm)
                loss = loss + CFG["aux_w"] * tF.binary_cross_entropy(pa, ya)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
        if y_te is not None:                                  # dev-only: print validation AUC per epoch
            from sklearn.metrics import roc_auc_score
            val_auc = roc_auc_score(y_te.ravel(), predict().ravel())
            print(f"   TCN seed {seed} epoch {ep + 1}/{epochs}: AUC {val_auc:.5f}", flush=True)
    return predict()

def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

## 5. Training

Training happens in two steps.

**Step 1 — temporal hold-out (only if `CFG["fit_blend"]` is True).** CatBoost, ExtraTrees and one TCN seed are trained on the older training cases and scored on the most recent 15 % of them. The boundary is chosen so that the fit part ends 7 days before the hold-out part: each case covers 7 days and a new case starts every day, so neighbouring cases overlap, and without this purge the models would see hours that also appear in the cases to predict (leakage). These hold-out predictions are used **only** in Section 6 to fit the blend weights and the smoothing kernel; the test cases are not touched.

**Step 2 — final models.** Every model is retrained on **all 2,922 training cases** (2015–2022) and predicts the 725 test cases (2023–2024). The TCN is trained once per seed (`CFG["tcn_seeds"]` = 3 on GPU, 2 on CPU) and the seed predictions are averaged.

Training is sequential: CatBoost first (GPU or CPU), then ExtraTrees (CPU, all threads), then the TCN. `CFG["fit_blend"] = False` skips Step 1 entirely (about +60 % running time) and uses the equal-weight constants in the blend cell.

In [ ]:
UH = U * H
X_tr, X_te = X[:n_tr_rows], X[n_tr_rows:]
preds, hold = {}, {}

def rows_of(cidx):
    """Feature rows of the given (sorted) case indices: a view when the cases are consecutive, a copy otherwise."""
    a, b = int(cidx[0]), int(cidx[-1]) + 1
    if len(cidx) == b - a:
        return X[a * UH:b * UH]
    return X[(cidx[:, None] * UH + np.arange(UH)).ravel()]

# ---- 1. Temporal hold-out: train on the older cases, predict the most recent ones ----------------------------------
# Used only to fit the blend weights and the smoothing below. The test cases are not touched here.
if CFG["fit_blend"]:
    sv = cases["start"].iloc[:n_train].dt.tz_convert(None).to_numpy()
    hold_start = np.sort(sv)[int(n_train * (1 - CFG["hold_frac"]))]
    hold_c = np.flatnonzero(sv >= hold_start)
    fit_c = np.flatnonzero(sv < hold_start - np.timedelta64(CFG["purge_days"], "D"))
    print(f"hold-out: {len(fit_c)} fit cases, {len(hold_c)} hold-out cases (from {str(hold_start)[:10]})", flush=True)
    Xf, yf = rows_of(fit_c), labels[fit_c].ravel()
    Xh, hold_y = rows_of(hold_c), labels[hold_c].ravel()

    t = time.time()
    hold["catboost"] = fit_predict_catboost(Xf, yf, Xh, CFG["cb_iters"])[0]
    print(f"[hold-out] catboost AUC {roc_auc_score(hold_y, hold['catboost']):.5f}, {time.time() - t:.0f}s", flush=True)
    t = time.time()
    hold["extratrees"] = fit_predict_et(Xf, yf, Xh)
    print(f"[hold-out] extratrees AUC {roc_auc_score(hold_y, hold['extratrees']):.5f}, {time.time() - t:.0f}s", flush=True)
    del Xf, yf, Xh

# ---- 2. Final models: all 2,922 training cases -> 725 test cases ---------------------------------------------------
t = time.time()
preds["catboost"] = fit_predict_catboost(X_tr, y_tr, X_te, CFG["cb_iters"])[0].reshape(-1, U, H)
print(f"catboost done, {time.time() - t:.0f}s", flush=True)

t = time.time()
preds["extratrees"] = fit_predict_et(X_tr, y_tr, X_te).reshape(-1, U, H)
print(f"extratrees done, {time.time() - t:.0f}s", flush=True)

t = time.time()
Z = prep_tensor(X, n_cases, n_tr_rows)
if CFG["fit_blend"]:
    hold["tcn"] = fit_predict_tcn(Z, labels, fit_c, hold_c, seed=100, epochs=CFG["tcn_epochs"]).ravel()
    print(f"[hold-out] sequence network AUC {roc_auc_score(hold_y, hold['tcn']):.5f}, {time.time() - t:.0f}s", flush=True)
tr_idx, te_idx = np.arange(n_train), np.arange(n_train, n_cases)
p_seeds = []
for s in range(CFG["tcn_seeds"]):
    p_seeds.append(fit_predict_tcn(Z, labels, tr_idx, te_idx, seed=s, epochs=CFG["tcn_epochs"]))
    print(f"sequence network seed {s} done, {time.time() - t:.0f}s", flush=True)
preds["tcn"] = np.mean(p_seeds, 0)
del Z


## 6. Blend

**Why logit space.** The three models produce probabilities on very different scales. Tree models (CatBoost, ExtraTrees) tend to push probabilities toward the extremes (0 or 1) because they minimise log-loss by being confident; neural-network sigmoids are often better calibrated but can be under-confident. Averaging the probabilities directly would let the tree models dominate numerically and dilute the neural network's information.

Instead, every probability is mapped to the log-odds (logit) scale:

$$z = \log\frac{p}{1-p}$$

a weighted average $\bar{z} = \sum_m w_m z_m + b$ is taken, and the result is mapped back:

$$p_{\text{blend}} = \sigma(\bar{z}) = \frac{1}{1 + e^{-\bar{z}}}$$

This treats over- and under-confidence symmetrically and allows a linear combination before the final sigmoid.

**How the weights are obtained.** With `CFG["fit_blend"]` True, the weights and the bias are fitted **inside this notebook** on the temporal hold-out predictions from Section 5, by a non-negative logistic regression on the three logits (L-BFGS-B on the average logistic loss, weights constrained to be non-negative, bias free). The fitted blend is compared with equal weights by hold-out AUC, and the better set is kept. Then a small smoothing kernel along the 168 hours (3-tap, 5-tap or a box filter) is applied to the blended logits **if and only if** it improves the hold-out AUC by a clear margin (`2e-5`); otherwise no smoothing is used.

**Fallback.** With `CFG["fit_blend"] = False` the constants at the top of this cell are used instead (equal weights by default); paste the output of a dev-notebook fit there if you have one.

In [ ]:
# Blend weights. With CFG["fit_blend"] they are fitted below on the temporal hold-out; otherwise paste the constants here.
BLEND = {"catboost": 1 / 3, "extratrees": 1 / 3, "tcn": 1 / 3}
BLEND_BIAS = 0.0
WEIGHTS_FITTED = False
SMOOTH_K = None                                  # smoothing kernel along the hours (None = no smoothing)

assert set(BLEND) == set(preds), "blend weights and trained models do not match"
MODELS = list(BLEND)

def smooth(z, k):
    """Weighted moving average of logits along the hour axis (edge-padded); z: (cases, U, H)."""
    if k is None:
        return z
    r = len(k) // 2
    zp = np.pad(z, ((0, 0), (0, 0), (r, r)), mode="edge")
    return sum(k[i] * zp[..., i:i + H] for i in range(len(k)))

KERNELS = {"none": None, "3-tap": np.array([1, 2, 1]) / 4, "5-tap": np.array([1, 2, 3, 2, 1]) / 9,
           "box-5": np.ones(5) / 5, "box-9": np.ones(9) / 9}

if CFG["fit_blend"]:
    Lh = np.stack([logit(hold[m]).astype(np.float64) for m in MODELS], 1)        # (hold-out rows, models)
    yh = hold_y.astype(np.float64)
    k = len(MODELS)

    def nll(w):                                   # logistic loss and its gradient
        z = Lh @ w[:k] + w[k]
        p = 1.0 / (1.0 + np.exp(-z))
        return np.mean(np.logaddexp(0, z) - yh * z), np.append(Lh.T @ (p - yh) / len(yh), np.mean(p - yh))

    res = minimize(nll, np.append(np.full(k, 1 / k), 0.0), jac=True, method="L-BFGS-B",
                   bounds=[(0, None)] * k + [(None, None)])
    w_fit, b_fit = res.x[:k], res.x[k]
    w_eq, b_eq = np.full(k, 1 / k), 0.0

    for m in MODELS:
        print(f"  hold-out AUC {m:>10}: {roc_auc_score(hold_y, hold[m]):.5f}")
    auc_eq = roc_auc_score(hold_y, Lh @ w_eq + b_eq)
    auc_fit = roc_auc_score(hold_y, Lh @ w_fit + b_fit)
    print(f"  hold-out AUC equal weights: {auc_eq:.5f} | fitted weights {dict(zip(MODELS, w_fit.round(3).tolist()))}, "
          f"bias {b_fit:.3f}: {auc_fit:.5f}")
    if auc_fit >= auc_eq:
        BLEND, BLEND_BIAS = dict(zip(MODELS, map(float, w_fit))), float(b_fit)
        z_hold = Lh @ w_fit + b_fit
    else:
        z_hold = Lh @ w_eq + b_eq
    WEIGHTS_FITTED = True

    # Smoothing along the hours: pick the kernel with the best hold-out AUC (must beat "none" by a clear margin)
    z_hold = z_hold.reshape(len(hold_c), U, H)
    scores = {name: roc_auc_score(hold_y, smooth(z_hold, kern).ravel()) for name, kern in KERNELS.items()}
    print("  hold-out AUC by smoothing kernel:", {n: round(s, 5) for n, s in scores.items()})
    best = max(scores, key=scores.get)
    if scores[best] - scores["none"] > 2e-5:
        SMOOTH_K = KERNELS[best]
    print("  chosen smoothing:", best if SMOOTH_K is not None else "none")
    del Lh, yh, z_hold
elif not WEIGHTS_FITTED:
    print("WARNING: the blend uses the equal-weight placeholder, not weights fitted on out-of-fold predictions.")

print("blend:", {m: round(w, 3) for m, w in BLEND.items()}, "bias", round(BLEND_BIAS, 3))
z = sum(BLEND[m] * logit(preds[m]) for m in BLEND) + BLEND_BIAS
z = smooth(z, SMOOTH_K)
p_blend = (1.0 / (1.0 + np.exp(-z))).astype(np.float32)
print({m: round(float(preds[m].mean()), 4) for m in preds}, "| blend mean", round(float(p_blend.mean()), 4),
      "| train ON rate", round(float(labels.mean()), 4))


## 7. Submission file

The $(725, 14, 168)$ blended probability array is turned back into the submission format: one row per test case (`Run No`) and one column per unit and hour (14 × 168 = 2,352 `result_committed_<unit>_t<hour>` columns), in the same column order as `sample_submission.csv`.

Before writing `submission.csv`, the code checks that:

- the file has exactly 725 rows and 2,353 columns (`Run No` + 2,352 unit-hour columns);
- all values are finite probabilities between 0 and 1;
- the `Run No` order matches `prediction_mapping.csv` exactly.

The last cell also prints the total runtime, to be copied into the hardware summary at the top of the notebook.

In [ ]:
sub = pd.DataFrame(p_blend.reshape(len(p_blend), -1), columns=LABEL_COLS)
sub.insert(0, "Run No", cases["Run No"].to_numpy()[n_train:])
sub = sub[pd.read_csv(SAMPLE_CSV, nrows=1).columns]       # enforce exact column order
vals = sub.iloc[:, 1:].to_numpy()
assert sub.shape == (725, 2353), f"wrong shape {sub.shape}"
assert np.isfinite(vals).all() and vals.min() >= 0 and vals.max() <= 1
assert (sub["Run No"].to_numpy() == mapping["Run No"].to_numpy()).all()
sub.to_csv(OUT_CSV, index=False)
total_h = (time.time() - T0) / 3600
print("Wrote", OUT_CSV.resolve(), sub.shape, f"| total runtime {total_h:.2f} h")
print("Please fill the hardware/runtime summary in the top markdown cell with the measured values above.")